# 05 - Modélisation : Régression Linéaire

Train/Test split + régression **simple** et **multiple** sur Lyon, Paris, Bordeaux.

**Régression linéaire** : trouve les coefficients qui minimisent la somme des erreurs au carré.
`prix = a₁×accommodates + a₂×bedrooms + ... + aₙ×featureₙ + b`

**2 modèles :**
- **Simple** : 1 variable (`accommodates`) → baseline de référence
- **Multiple** : 18 variables → on ajoute toutes les features préparées

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CITIES = {'Lyon':'lyon','Paris':'paris','Bordeaux':'bordeaux'}
COLORS = {'Lyon':'steelblue','Paris':'coral','Bordeaux':'mediumseagreen'}
TARGET = 'price'
dfs = {n: pd.read_csv(f'../data/processed/{k}_features.csv') for n, k in CITIES.items()}
print({k: df.shape for k, df in dfs.items()})

## 7. Séparation Train / Test

**Principe :** on ne peut pas évaluer un modèle sur les données qu'il a vues à l'entraînement.

- **80% → train** : le modèle apprend
- **20% → test** : on évalue sur des données **jamais vues**

`random_state=42` : graine aléatoire fixe → résultats reproductibles à chaque exécution.

`fit_eval()` encapsule tout le pipeline : dropna → split → fit → predict → métriques.

In [ ]:
SIMPLE_FEAT = ['accommodates']  # 1 seule variable pour la régression simple

MULTIPLE_FEAT = [
    'accommodates','bedrooms','beds','bathrooms','minimum_nights',
    'availability_365','number_of_reviews','reviews_per_month',
    'review_scores_rating','review_scores_cleanliness','review_scores_location',
    'host_is_superhost','host_response_rate','calculated_host_listings_count',
    'instant_bookable','room_type_code','neighbourhood_freq','property_type_freq',
]

def fit_eval(df, features):
    """Pipeline : prépare, entraîne et évalue un modèle LinearRegression."""
    feats = [f for f in features if f in df.columns]
    d = df[feats + [TARGET]].dropna()  # scikit-learn ne tolère pas les NaN
    X, y = d[feats], d[TARGET]
    # 80/20 split, reproductible avec random_state=42
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    m = LinearRegression().fit(Xtr, ytr)  # calcule les coefficients optimaux
    yp = m.predict(Xte)  # prédit sur les données jamais vues
    mae  = round(mean_absolute_error(yte, yp), 2)
    rmse = round(np.sqrt(mean_squared_error(yte, yp)), 2)
    r2   = round(r2_score(yte, yp), 4)
    return m, Xte, yte, yp, feats, mae, rmse, r2

results = {}
print('=== Entraînement ===')
for name, df in dfs.items():
    print(f'\n{name}  (train={int(len(df)*0.8):,}  test={int(len(df)*0.2):,})')
    results[f'{name}_simple']   = fit_eval(df, SIMPLE_FEAT)
    results[f'{name}_multiple'] = fit_eval(df, MULTIPLE_FEAT)
    for label in ['simple','multiple']:
        r = results[f'{name}_{label}']
        print(f'  {label:8s} -> MAE={r[5]}€  RMSE={r[6]}€  R²={r[7]}')

## 7a. Régression Linéaire Simple

Variable unique : `accommodates` — la plus corrélée au prix (vue dans notebook 03).

**Graphique :** nuage de points (prix réel vs nb personnes) + droite de régression.
Les points très éloignés de la droite = erreurs du modèle (résidus élevés).

**Limite :** un studio pour 2 personnes au centre de Paris n'a pas le même prix
qu'un logement pour 2 en banlieue → d'où la nécessité de la régression multiple.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name in zip(axes, CITIES):
    m, Xte, yte, yp, feats, mae, rmse, r2 = results[f'{name}_simple']
    ax.scatter(Xte['accommodates'], yte, alpha=0.25, s=8, color=COLORS[name], label='Réel')
    xs = sorted(Xte['accommodates'].unique())
    ax.plot(xs, m.predict(pd.DataFrame({'accommodates':xs})),
            color='black', linewidth=2, label='Droite')
    ax.set_title(f'Simple - {name} | MAE={mae}€  R²={r2}')
    ax.set_xlabel('Nb personnes'); ax.set_ylabel('Prix (€)'); ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/regression_simple.png', bbox_inches='tight')
plt.show()

## 7b. Régression Linéaire Multiple

**Graphique prix prédit vs prix réel** :
- Diagonale pointillée = prédiction parfaite
- Points proches de la diagonale = bonnes prédictions
- Points loin = erreurs

**R² multiple > R² simple** : les 17 features supplémentaires apportent
une information réelle et réduisent l'erreur.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name in zip(axes, CITIES):
    m, Xte, yte, yp, feats, mae, rmse, r2 = results[f'{name}_multiple']
    ax.scatter(yte, yp, alpha=0.2, s=8, color=COLORS[name])
    lims = [min(yte.min(),yp.min()), max(yte.max(),yp.max())]
    ax.plot(lims, lims, 'k--', linewidth=1, label='Prédiction parfaite')
    ax.set_title(f'Multiple - {name} | MAE={mae}€  R²={r2}')
    ax.set_xlabel('Prix réel (€)'); ax.set_ylabel('Prix prédit (€)'); ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/regression_multiple.png', bbox_inches='tight')
plt.show()

## Importance des coefficients

Chaque coefficient indique l'impact d'une feature sur le prix prédit.
- **Barre positive** : cette feature augmente le prix
- **Barre négative** : cette feature diminue le prix
- **Longueur** : force de l'impact

**Attention :** les coefficients ne sont pas comparables directement entre features
car les unités sont différentes (`accommodates` en personnes, `neighbourhood_freq` entre 0 et 1).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, name in zip(axes, CITIES):
    m, _, _, _, feats, _, _, _ = results[f'{name}_multiple']
    # pd.Series associe chaque coefficient à son nom de feature
    coefs = pd.Series(m.coef_, index=feats).sort_values()
    ax.barh(coefs.index, coefs.values,
            color=['coral' if v < 0 else COLORS[name] for v in coefs])
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Coefficients - {name}'); ax.set_xlabel('Coefficient')
plt.tight_layout()
plt.savefig('../reports/figures/coefficients.png', bbox_inches='tight')
plt.show()